# 238. Product of Array Except Self
**Difficulty:** 🟡 Medium · **Topic:** Array · **LeetCode:** https://leetcode.com/problems/product-of-array-except-self/

## 💡 Concepts

**Core concept(s):** **Prefix and suffix products** — for each index, the answer is (product of everything to its left) × (product of everything to its right).

**Why it applies here:** "Product except self" splits cleanly into two independent parts around each index. If we precompute the running product from the left and from the right, each answer is a single multiplication — no division needed (which matters because the problem forbids division and division breaks on zeros).

**Key intuition / mental model:** Two sweeps. First pass fills each slot with the product of all elements before it. Second pass multiplies in the product of all elements after it.

---

### 📚 What is a Prefix / Suffix Product (or Prefix Sum)?
A **prefix product** `L[i]` is the product of `nums[0..i-1]`; a **suffix product** `R[i]` is the product of `nums[i+1..]`. (The additive version is the ubiquitous **prefix sum**.) They let range queries collapse to O(1) after an O(n) precompute.
- **Complexity:** O(n) to build, O(1) per lookup.
- **In Python:** a list you fill with a running variable, or `itertools.accumulate`.

## 📝 Problem

Return an array `out` where `out[i]` is the product of **all** elements of `nums` except `nums[i]`. Do it **without division** and in O(n).

**Example**
```
Input:  nums = [1, 2, 3, 4]
Output: [24, 12, 8, 6]
```
**Constraints:** `2 <= len(nums) <= 10^5`; the full product fits in a 32-bit int.

### Approach 1 — Brute Force (worst)

**Idea:** For each `i`, multiply every other element with an inner loop.

**Time complexity:** `O(n^2)`.

**Space complexity:** `O(1)` extra (besides output).

In [ ]:
from typing import List

def product_except_self_brute(nums: List[int]) -> List[int]:
    n = len(nums)
    out = [1] * n
    for i in range(n):                     # compute the answer for each position i
        p = 1
        for j in range(n):                 # multiply together every OTHER element
            if j != i:                     # skip the element at i itself
                p *= nums[j]
        out[i] = p
    return out

### Approach 2 — Division (better, but disallowed here)

**Idea:** Multiply everything, then divide by `nums[i]`. Shown for contrast — the problem forbids division, and it needs a special case for zeros. If exactly one zero exists, only that slot is nonzero; if two or more zeros, all slots are zero.

**Time complexity:** `O(n)`.

**Space complexity:** `O(1)` extra.

In [ ]:
from typing import List

def product_except_self_division(nums: List[int]) -> List[int]:
    zeros = nums.count(0)
    if zeros > 1:
        return [0] * len(nums)             # two or more zeros -> every product is 0
    prod = 1
    for x in nums:
        if x != 0:
            prod *= x                      # product of all NON-zero values
    if zeros == 1:                         # exactly one zero: only its slot survives
        return [prod if x == 0 else 0 for x in nums]
    return [prod // x for x in nums]       # no zeros -> divide total by each value

### Approach 3 — Prefix × Suffix Products (optimal, no division)

**Idea:** First pass: `out[i]` = product of everything **left** of `i`. Second pass: walk right-to-left with a running suffix product and multiply it in. Output array doubles as scratch, so extra space is O(1).

**Time complexity:** `O(n)` — two passes.

**Space complexity:** `O(1)` extra (output not counted).

In [ ]:
from typing import List

def product_except_self_optimal(nums: List[int]) -> List[int]:
    n = len(nums)
    out = [1] * n
    prefix = 1                             # running product of everything to the LEFT
    for i in range(n):
        out[i] = prefix                    # store product of all elements before i
        prefix *= nums[i]                  # extend the running left-product
    suffix = 1                             # running product of everything to the RIGHT
    for i in range(n - 1, -1, -1):         # sweep right to left
        out[i] *= suffix                   # multiply in the product of all elements after i
        suffix *= nums[i]                  # extend the running right-product
    return out

In [ ]:
# Correctness check
tests = [
    ([1, 2, 3, 4], [24, 12, 8, 6]),
    ([-1, 1, 0, -3, 3], [0, 0, 9, 0, 0]),
    ([2, 3], [3, 2]),
]
for nums, expected in tests:
    b = product_except_self_brute(nums)
    d = product_except_self_division(nums)
    o = product_except_self_optimal(nums)
    print(f"{nums} -> brute={b}, division={d}, optimal={o} | expected={expected}")
    assert b == d == o == expected, "mismatch!"
print("\nAll tests passed")

## ⏱️ Empirically Checking the Complexities

Big-O can't be read off a function directly, but it can be **measured**. We time each approach on inputs of growing `n` and read the **doubling ratio** — how much runtime grows when `n` doubles.

| Theoretical | Ratio when `n` → `2n` |
|-------------|-----------------------|
| `O(log n)`    | ≈ **1×** |
| `O(n)`        | ≈ **2×** |
| `O(n log n)`  | ≈ **2×** (slightly more) |
| `O(n²)`       | ≈ **4×** |
| `O(n³)`       | ≈ **8×** |

Inputs are built to force the **worst case** (no early exit) so the measurement reflects the true bound. Sub-millisecond rows are noisy — look at the trend, not one number.

In [ ]:
import os, sys
_root = os.getcwd()
for _ in range(5):
    if os.path.exists(os.path.join(_root, "bench_utils.py")):
        break
    _root = os.path.dirname(_root)
if _root not in sys.path:
    sys.path.insert(0, _root)
from bench_utils import benchmark   # shared: prints ratio table + optional log-log plot

def make_worst_case(n):
    # Values in {+1, -1}: no zeros, yet products stay bounded so each multiply is
    # O(1) -- otherwise factorial-sized big-ints would mask the true O(n)/O(n^2) shape.
    nums = [1 if i % 2 == 0 else -1 for i in range(n)]
    return (nums,)

solutions = {
    "brute    O(n^2)": product_except_self_brute,
    "division O(n)  ": product_except_self_division,
    "optimal  O(n)  ": product_except_self_optimal,
}
sizes = [1000, 2000, 4000, 8000]

benchmark(solutions, make_worst_case, sizes, plot=True)


## 🧩 Patterns Learned

- **Prefix/suffix precomputation:** When each answer depends on "everything on one side", precompute directional running aggregates and combine in O(1).
- **Avoiding division:** Prefix×suffix sidesteps division entirely — essential when division is disallowed or when zeros make it unsafe.
- **Signal to reach for it:** "product/sum of all except i", "range sum/product queries", "leftmost/rightmost aggregate".
- **Related problems:** Range Sum Query (prefix sums), Trapping Rain Water, Maximum Subarray, Subarray Sum Equals K.
- **Common pitfalls:** (1) using division without handling zeros; (2) allocating separate prefix and suffix arrays when the output array can be reused; (3) off-by-one in the reverse pass.